# Cattle Re-ID — Gap crudo cross-dataset MISMA modalidad (CMPD300 → Zenodo, hocico→hocico)

Opción 2: en vez de hocico→cara (que no dio gap útil), medimos el gap **hocico→hocico entre dos
datasets distintos**: encoder entrenado en CMPD300 (Fase 5), evaluado sobre el dataset de hocicos
de Zenodo (el del paper), **sin adaptar**. El shift es de cámara/país/raza sobre la MISMA
modalidad → el encoder de hocico es lo relevante y debería haber gap medible.

Reusa el mismo harness de la Fase 6 (`06_eval_reid.py`), solo cambia el `--target-dir` y usa
split normal por identidad (sin `--by-session`: Zenodo no tiene fotos gemelas).

### Prerrequisitos (en Drive `MyDrive/cattle_reid/`)
- `CMPD300_Baseline.zip` (para el sanity) y `BeefCattle_Muzzle_Individualized.zip` (target Zenodo).
- `outputs/checkpoints/cmpd300_source.pt` (encoder de la Fase 5).
- El código de Fase 6 commiteado en el repo (fork, `rama_trini1`).

## 0. GPU

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'Sin GPU: Entorno de ejecución -> Cambiar tipo -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')
assert DRIVE_ROOT.is_dir(), f'No existe {DRIVE_ROOT}.'
print('DRIVE_ROOT:', DRIVE_ROOT)

## 2. Traer el código (clon limpio de tu fork, rama_trini1)

In [ ]:
import os, shutil
os.chdir('/content')
REPO_URL = 'https://github.com/trinidadpujol/tp-final-vision2-Pujol-Vitale.git'
REPO_DIR = '/content/tp-final-vision2-Pujol-Vitale'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -b rama_trini1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

## 3. Verificar código + checkpoint

In [ ]:
!python scripts/06_eval_reid.py \
    --source-dir "{SOURCE_DIR}" \
    --target-dir "{TARGET_DIR}" \
    --single-shot \
    --compare-imagenet

## 4. Persistir outputs en Drive (trae el cmpd300_source.pt de la Fase 5)

In [ ]:
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
for sub in ('checkpoints', 'logs', 'results'):
    drive_sub = DRIVE_OUTPUTS / sub; drive_sub.mkdir(parents=True, exist_ok=True)
    local_sub = Path(REPO_DIR) / 'outputs' / sub
    if local_sub.exists() and not local_sub.is_symlink():
        shutil.rmtree(local_sub)
    if not local_sub.exists():
        os.symlink(drive_sub, local_sub, target_is_directory=True)
assert (Path(REPO_DIR)/'outputs/checkpoints/cmpd300_source.pt').is_file(), 'no veo el checkpoint'
print('checkpoint accesible ✅')

## 5. CMPD300 desde Drive (para el sanity)

In [ ]:
import zipfile
CMPD_ZIP = DRIVE_ROOT / 'CMPD300_Baseline.zip'
assert CMPD_ZIP.is_file(), f'Falta {CMPD_ZIP}.'
CMPD_LOCAL = Path('/content/cmpd300')
if not CMPD_LOCAL.exists():
    CMPD_LOCAL.mkdir(parents=True)
    !unzip -q "{CMPD_ZIP}" -d "{CMPD_LOCAL}"
cand = [CMPD_LOCAL, CMPD_LOCAL/'Baseline'] + [p for p in CMPD_LOCAL.iterdir() if p.is_dir()]
CMPD_DIR = next((p for p in cand if (p/'train').is_dir()), None)
assert CMPD_DIR, 'No encuentro train/ en CMPD300.'
SOURCE_DIR = CMPD_DIR / 'train'
print('SOURCE_DIR (sanity):', SOURCE_DIR)

## 6. Zenodo (hocicos) desde Drive — TARGET

Dataset del paper (~640 MB, 268 individuos). Se extrae completo y se ubica la carpeta que
contiene las subcarpetas por individuo (cattle_XXXX).

In [ ]:
ZEN_ZIP = DRIVE_ROOT / 'BeefCattle_Muzzle_Individualized.zip'
assert ZEN_ZIP.is_file(), (f'Falta {ZEN_ZIP}. Es el dataset de hocicos de Zenodo de la Etapa 1; '
                           'subilo a Drive como zip.')
ZEN_LOCAL = Path('/content/zenodo')
if not ZEN_LOCAL.exists():
    ZEN_LOCAL.mkdir(parents=True)
    !unzip -q "{ZEN_ZIP}" -d "{ZEN_LOCAL}"

# ubicar la carpeta con muchas subcarpetas de individuos
TARGET_DIR = None
cands = list(ZEN_LOCAL.rglob('BeefCattle_Muzzle_Individualized')) + [ZEN_LOCAL] + \
        [p for p in ZEN_LOCAL.iterdir() if p.is_dir()]
for c in cands:
    if c.is_dir() and sum(1 for p in c.iterdir() if p.is_dir()) >= 50:
        TARGET_DIR = c; break
assert TARGET_DIR, 'No encuentro la carpeta con subcarpetas por individuo.'
n_ids = sum(1 for p in TARGET_DIR.iterdir() if p.is_dir())
print(f'TARGET_DIR (Zenodo): {TARGET_DIR} | {n_ids} individuos')

## 7. Correr el gap (CMPD300 → Zenodo, split normal por identidad)

Sin `--by-session` (Zenodo no tiene fotos gemelas). Con `--compare-imagenet` para confirmar que
acá SÍ el encoder de hocico le gana a ImageNet.

In [ ]:
!python scripts/06_eval_reid.py \
    --source-dir "{SOURCE_DIR}" \
    --target-dir "{TARGET_DIR}" \
    --single-shot \
    --compare-imagenet

## 8. Resultados

In [ ]:
import json
p = Path('outputs/results/06_reid_summary.json')
print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False))

## Cómo leer esto (SINGLE-SHOT)

Single-shot: 1 sola imagen por individuo en gallery → reduce la fuga por fotos parecidas.

- **Sanity CMPD300** (~0.98): harness OK.
- **encoder de hocico vs ImageNet PURO** (mismo split): la clave. Si tu encoder ahora le **saca
  ventaja clara** a ImageNet → hay biometría de hocico real y el número es confiable; la caída
  respecto al sanity (~0.98) es el **domain gap** para el DA. Si **siguen empatados** aun en
  single-shot → el encoder de hocico no aporta sobre features genéricas, y el problema es de los
  datos (fotos demasiado parecidas por individuo) → decisión con Gastón.